# Preprocessing NN Data Figures 4 and 5: 4D-Var State Estimation and Analysis Skill

In [1]:
%matplotlib inline
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns

import numpy as np
import pandas as pd
import glob

import joblib


In [8]:
# NN-only background-state results
nn_base_dir = "../../data/nn_assimilation_results"

nn_xb_files = {
    "baseline": "two_samples_per_day_nn_npz_xb_data_10000xb_estimates_min_guess_0.05_5_512_compressed.pkl",
    "irreg_sparse": "irregular_sparse_nn_npz_xb_data_10000xb_estimates_min_guess_0.05_5_512_compressed.pkl",
    "field_informed": "field_informed_nn_npz_xb_data_10000xb_estimates_min_guess_0.05_5_512_compressed.pkl",
}

# Load NN background-state data
nn_xb_dicts = {
    name: joblib.load(os.path.join(nn_base_dir, fname))
    for name, fname in nn_xb_files.items()
}

# Preprocess background-state distances
rows = []

for dataset_name, nn_xb_dict in nn_xb_dicts.items():
    for i, record in nn_xb_dict.items():
        perturbed = record["perturbed_initial_state"]
        truth = record["truth"][0]
        diff = perturbed - truth

        rows.append({
            "dataset": dataset_name,
            "index": i,
            "N_perturbed": perturbed[0],
            "P_perturbed": perturbed[1],
            "Z_perturbed": perturbed[2],
            "distance_N": np.abs(diff[0]),
            "distance_P": np.abs(diff[1]),
            "distance_Z": np.abs(diff[2]),
            "distance_total_L2": np.linalg.norm(diff),
            "distance_total_L1": np.sum(np.abs(diff)),
        })

df_nn_xb_distances = pd.DataFrame(rows)

print(df_nn_xb_distances.shape)
display(df_nn_xb_distances.head())

(30000, 10)


,dataset,index,N_perturbed,P_perturbed,Z_perturbed,distance_N,distance_P,distance_Z,distance_total_L2,distance_total_L1
0,baseline,0,1.668613,0.324782,0.056039,0.068613,0.024782,0.043961,0.085173,0.137356
1,baseline,1,1.752258,0.259964,0.134752,0.152258,0.040036,0.034752,0.161224,0.227045
2,baseline,2,1.617352,0.284119,0.095992,0.017352,0.015881,0.004008,0.023861,0.037241
3,baseline,3,1.628221,0.319747,0.098054,0.028221,0.019747,0.001946,0.034498,0.049914
4,baseline,4,1.671907,0.245825,0.169676,0.071907,0.054175,0.069676,0.113843,0.195758


In [10]:
# Original PI-NPZ / RK4 background data
original_base_dir = "../../data/assimilation_results"

min_val = "0.05"
suffix = "_frozen_params_5_384_compressed.pkl"

original_xb_files = {
    "baseline": f"two_samples_per_day_xb_data_10000xb_estimates_min_guess_{min_val}{suffix}",
    "irreg_sparse": f"irregular_sparse_xb_data_10000xb_estimates_min_guess_{min_val}{suffix}",
    "field_informed": f"field_informed_xb_data_10000xb_estimates_min_guess_{min_val}{suffix}",
}

original_xb_dicts = {
    name: joblib.load(os.path.join(original_base_dir, fname))
    for name, fname in original_xb_files.items()
}

In [11]:
# Use baseline experiment
nn_xb_dict = nn_xb_dicts["baseline"]
original_xb_dict = original_xb_dicts["baseline"]

# --- Collect NN predictions and RK4 truth ---
nn_preds = []
rk4_truths = []

for i, record in nn_xb_dict.items():
    nn_preds.append(
        np.array(record["nn_npz_background_state"])
    )
    rk4_truths.append(
        np.array(original_xb_dict[i]["rk4_background_state"])
    )

nn_preds = np.array(nn_preds)      # shape [N, T, 3]
rk4_truths = np.array(rk4_truths)  # shape [N, T, 3]

# --- Compute NN skill vs RK4 truth ---
nn_scores = 1 - (
    np.abs(nn_preds - rk4_truths) /
    np.abs(rk4_truths)
)

# Mean/std over all trajectories
nn_mean = np.nanmean(nn_scores, axis=0).T
nn_std = np.nanstd(nn_scores, axis=0).T

# --- Summary statistics ---
skill_mean_over_time = np.nanmean(nn_mean, axis=1)
skill_std_over_time = np.nanmean(nn_std, axis=1)

global_skill_mean = np.nanmean(nn_mean)
global_skill_std = np.nanmean(nn_std)

traj_skill_means = np.nanmean(nn_scores, axis=(1, 2))
traj_skill_stds = np.nanstd(nn_scores, axis=(1, 2))

early_skill = np.nanmean(nn_mean[:, :50], axis=1)
late_skill = np.nanmean(nn_mean[:, -50:], axis=1)

summary_df = pd.DataFrame({
    "Variable": ["N", "P", "Z"],
    "Mean Skill": skill_mean_over_time,
    "Std Skill": skill_std_over_time,
    "Early Skill (first 50 steps)": early_skill,
    "Late Skill (last 50 steps)": late_skill
})

display(summary_df)

,Variable,Mean Skill,Std Skill,Early Skill (first 50 steps),Late Skill (last 50 steps)
0,N,0.991646,0.004147,0.998752,0.992141
1,P,0.991243,0.003904,0.995289,0.973386
2,Z,0.991043,0.005137,0.997288,0.993745


In [12]:
# === NN-NPZ assimilation preprocessing ===

min_val = "0.05"

base_path = "../../data/nn_assimilation_results"

file_info = [
    (
        "baseline",
        "NN-NPZ",
        f"two_samples_per_day_nn_npz_frozen_assimilation_10000xb_estimates_min_guess_{min_val}_2000_5_512_compressed.pkl",
    ),
    (
        "irreg_sparse",
        "NN-NPZ",
        f"irregular_sparse_nn_npz_frozen_assimilation_10000xb_estimates_min_guess_{min_val}_2000_5_512_compressed.pkl",
    ),
    (
        "field_informed",
        "NN-NPZ",
        f"field_informed_nn_npz_frozen_assimilation_10000xb_estimates_min_guess_{min_val}_2000_5_512_compressed.pkl",
    ),
]

# === Build NN-NPZ DataFrame ===
records = []

for dataset_label, method, filename in file_info:
    path = os.path.join(base_path, filename)

    if not os.path.exists(path):
        print(f"Missing file: {path}")
        continue

    data = joblib.load(path)

    for i, result in data.items():
        record = {
            "dataset": dataset_label,
            "method": method,
            "index": i,

            # Cost-function components
            "jo_b": result.get("jo_b_nn_npz"),
            "jb_b": result.get("jb_b_nn_npz"),
            "J_total_b": result.get("J_total_b_nn_npz"),
            "jo_a": result.get("jo_a_nn_npz"),
            "jb_a": result.get("jb_a_nn_npz"),
            "J_total_a": result.get("J_total_a_nn_npz"),

            # Misfits
            "misfitb": result.get("nn_npz_misfitb"),
            "misfita": result.get("nn_npz_misfita"),

            # Cost-function improvement
            "Improvement": result.get("Improvement_nn_npz"),
            "pct_drop_J": result.get("pct_drop_J_nn_npz"),
            "pct_drop_Jo": result.get("pct_drop_Jo_nn_npz"),
            "pct_drop_Jb": result.get("pct_drop_Jb_nn_npz"),

            # Optimization diagnostics
            "cg_iterations": result.get("cg_iterations_nn_npz"),
            "converged": result.get("converged_nn_npz"),
            "exit_code": result.get("exit_code_nn_npz"),

            # Runtime
            "runtime": result.get("nn_npz_time"),

            # State estimates
            "xa": result.get("xa_nn_npz"),
            "xb": result.get("xb_nn_npz"),
        }

        records.append(record)

df_nn_flat = pd.DataFrame.from_records(records)

print(df_nn_flat.shape)
display(df_nn_flat.head())

(30000, 21)


,dataset,method,index,jo_b,jb_b,J_total_b,jo_a,jb_a,J_total_a,misfitb,...,Improvement,pct_drop_J,pct_drop_Jo,pct_drop_Jb,cg_iterations,converged,exit_code,runtime,xa,xb
0,baseline,NN-NPZ,0,477.348117,1.381587e-27,238.674058,301.254307,17.698290,159.476299,3.820900,...,19.862667,0.331824,0.368900,-1.281011e+28,77,1,0,1.398906,"[[1.6591128351192606, 0.321105606935122, 0.063...","[[1.668612793065758, 0.3247818079051125, 0.056..."
1,baseline,NN-NPZ,1,1517.827343,0.000000e+00,758.913672,1037.159334,38.619540,537.889437,4.915339,...,18.623724,0.291238,0.316682,NaN,55,1,0,1.021544,"[[1.7245218513138831, 0.26806572701461434, 0.1...","[[1.7522580799121814, 0.25996416707067627, 0.1..."
2,baseline,NN-NPZ,2,111.830944,1.369550e-27,55.915472,69.458639,2.373735,35.916187,1.237462,...,29.695639,0.357670,0.378896,-1.733222e+27,67,1,0,1.248092,"[[1.6153200781818011, 0.2899959643412966, 0.09...","[[1.617351963652655, 0.2841186855667228, 0.095..."
3,baseline,NN-NPZ,3,223.128648,4.814825e-29,111.564324,144.224183,1.593757,72.908970,1.679077,...,20.412154,0.346485,0.353628,-3.310103e+28,56,1,0,1.002845,"[[1.6234946047758863, 0.3159823710939789, 0.09...","[[1.6282208210798061, 0.31974691465168925, 0.0..."
4,baseline,NN-NPZ,4,1027.870450,0.000000e+00,513.935225,693.809296,50.989869,372.399582,4.887513,...,17.773994,0.275396,0.325003,NaN,55,1,0,0.991196,"[[1.6548458059558169, 0.25451191543598717, 0.1...","[[1.6719069382304, 0.24582477071720413, 0.1696..."


In [18]:
# Check NN-NPZ convergence across all experiments
print("Total experiments:", len(df_nn_flat))

print("\nConverged:")
print(df_nn_flat["converged"].value_counts(dropna=False))

print("\nExit codes:")
print(df_nn_flat["exit_code"].value_counts(dropna=False).sort_index())

print("\nCG iterations:")
print(df_nn_flat["cg_iterations"].describe())

print("\nMaximum CG iterations:",
      df_nn_flat["cg_iterations"].max())

print("\nRuns reaching 2000 iterations:",
      (df_nn_flat["cg_iterations"] >= 2000).sum())

print("\nConvergence by scenario:")
print(
    df_nn_flat.groupby("dataset").agg(
        n=("converged", "size"),
        converged=("converged", "sum"),
        max_iterations=("cg_iterations", "max"),
        mean_iterations=("cg_iterations", "mean"),
    )
)

Total experiments: 30000

Converged:
converged
1    30000
Name: count, dtype: int64

Exit codes:
exit_code
0    30000
Name: count, dtype: int64

CG iterations:
count    30000.000000
mean        64.225567
std         15.214912
min         44.000000
25%         51.000000
50%         58.000000
75%         77.000000
max        149.000000
Name: cg_iterations, dtype: float64

Maximum CG iterations: 149

Runs reaching 2000 iterations: 0

Convergence by scenario:
                    n  converged  max_iterations  mean_iterations
dataset                                                          
baseline        10000      10000              92          59.2191
field_informed  10000      10000              97          50.8921
irreg_sparse    10000      10000             149          82.5655


In [15]:
# Add a log-transformed runtime column before melt
# === Derived quantities for NN-NPZ ===
df_nn_flat["log_runtime"] = np.log10(df_nn_flat["runtime"])

df_nn_flat["jo_jb_ratio"] = (
    df_nn_flat["jo_a"] / df_nn_flat["jb_a"]
)

df_nn_flat["log_jo_jb_ratio"] = np.log10(
    df_nn_flat["jo_jb_ratio"]
)

df_nn_flat["pct_drop_J_100"] = (
    100 * df_nn_flat["pct_drop_J"]
)

# === Plotting-ready NN-NPZ data ===
metrics_normal = [
    "runtime",
    "pct_drop_J_100",
    "log_jo_jb_ratio"
]

df_nn_long_normal = df_nn_flat.melt(
    id_vars=["method", "dataset"],
    value_vars=metrics_normal,
    var_name="metric",
    value_name="value"
)

df_nn_long_normal["metric"] = df_nn_long_normal["metric"].replace({
    "runtime": "Computational Time (s)",
    "pct_drop_J_100": "Cost Function Reduction (%)",
    "log_jo_jb_ratio": "log$_{10}(J_o/J_b)$"
})



In [17]:
# Save NN assimilation metrics without trajectory arrays
df_nn_metrics = df_nn_flat.drop(columns=["xa", "xb"])

# Output directory for processed NN-NPZ data
output_dir = "../../data/nn_processed_assimilation_data"
os.makedirs(output_dir, exist_ok=True)

# Save NN assimilation metrics without trajectory arrays
df_nn_metrics = df_nn_flat.drop(columns=["xa", "xb"])

df_nn_metrics.to_csv(
    os.path.join(output_dir, "nn_assimilation_metrics.csv"),
    index=False
)

# Save plotting-ready format
df_nn_long_normal.to_csv(
    os.path.join(output_dir, "nn_assimilation_metrics_long.csv"),
    index=False
)